# Phần 9 — Inference, calibration và latency trên ConvNeXt-Tiny T04
**2A202602810 · Nguyễn Thái Anh**

T04 (CutMix) là ứng viên training tốt nhất của phần 8: macro-F1 val **97,5457%**.
Dùng đúng best.pt của T04/seed0 cho I00, I01, I02, I07 và I08; chỉ đánh giá validation.
Checkpoint/recipe/split/protocol được lưu hash; chưa khóa cấu hình chung kết hoặc mở test.

Chọn kernel `.venv` trên host GPU. Runner chạy process con để giải phóng GPU khi kết thúc.
Chạy lại cùng SESSION_ID đọc kết quả đã hoàn tất và tiếp tục các phần còn thiếu;
không đổi config/checkpoint/protocol giữa chừng. Không chạy train đồng thời với benchmark.

## 1. Môi trường và nguồn checkpoint

In [ ]:
import os, sys, json
from pathlib import Path
ROOT_HINT = None
ROOT = Path(ROOT_HINT).expanduser().resolve() if ROOT_HINT else next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (p / "eval.py").is_file() and (p / "starter").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Đặt ROOT_HINT tới gốc repo")
CODE = ROOT / "submissions/2A202602810_NguyenThaiAnh/code"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ.setdefault("MPLCONFIGDIR",str(ROOT / ".cache/matplotlib"))
sys.path.insert(0,str(ROOT)); sys.path.insert(0,str(CODE))
import pandas as pd
from IPython.display import display, Image, Markdown
import inference_notebook as runner
import train as trainer
cfg, source_summary, metadata = runner.source_context()
assert Path(trainer.__file__).resolve() == CODE / "train.py"
print("Nguồn:",trainer.run_dir(cfg) / "best.pt")
print("Best epoch:",source_summary["best_epoch"])
print("Macro-F1 val T04:",source_summary["val"]["macro_f1"])
print("Python:",sys.executable)

## 2. Cấu hình chạy
RUN_QUALITY=True: forward đầy đủ validation cho các phương pháp còn thiếu.
RUN_LATENCY=True: đo GPU riêng, warmup 10 + 100 lượt, batch 1 và 32.
Đặt cả hai False để chỉ xem kết quả đã có. Các flag không thay đổi protocol.
SESSION_ID mới khi muốn đo lại với protocol khác; không ghi đè benchmark cũ.

In [ ]:
SESSION_ID = "I_T04_seed0"
RUN_QUALITY = True
RUN_LATENCY = True
DEVICE = "cuda:0"
VAL_BATCH = 64
THROUGHPUT_BATCH = 32
NUM_WORKERS = 2
WARMUP = 10
ITERS = 100
print("Output:",runner.session_paths(SESSION_ID))
display(pd.DataFrame([{"method":m,"description":v[0],"K":v[1],"dtype":v[2]} for m,v in runner.METHODS.items()]))

## 3. Các view và phạm vi benchmark
I00: center crop 224 trên nguồn 256. I01: center crop và bản lật ngang, mean probability.
I02: bốn góc + giữa, **5 vị trí crop khác nhau** trên ảnh nguồn 256; không crop từ ảnh đã là 224.
I07: fit scalar T trên logits I00 của toàn bộ val, tối thiểu NLL; accuracy/F1 giữ nguyên.
I08: CUDA FP16 autocast, softmax/aggregation giữ FP32. ConvNeXt dùng LayerNorm nên fusion BN không áp dụng.

Latency gồm GPU crop/views, forward, gộp/calibration và softmax trên tensor 256 đã ở GPU.
Bỏ disk IO, resize/normalize CPU, H2D và fit T offline; đây chưa phải latency end-to-end robot.
Có sync GPU trước/sau mỗi sample, >=10 warmup và >=50 sample; giữ các sample ms để audit.
Runner từ chối đo khi có process compute GPU khác. Kiểm tra tải desktop và đóng các job train trước.

In [ ]:
if RUN_QUALITY or RUN_LATENCY:
    import subprocess
    query = subprocess.run(["nvidia-smi","--query-gpu=name,memory.free,utilization.gpu","--format=csv"],capture_output=True,text=True)
    print(query.stdout if query.returncode == 0 else query.stderr)
print("Phạm vi:",runner.SCOPE)
display(runner.collect_session(SESSION_ID))

## 4. Chạy validation và benchmark
Process con stream log theo phương pháp và batch. Không cập nhật model, không tải pretrained,
không tạo test loader. I00 phải khớp logits/probabilities đã lưu của checkpoint T04.
Interrupt dừng process con; chạy lại giữ SESSION_ID để tiếp tục các phương pháp chưa hoàn tất.

In [ ]:
if RUN_QUALITY or RUN_LATENCY:
    frame = runner.run_in_subprocess(SESSION_ID,run_quality=RUN_QUALITY,run_latency=RUN_LATENCY,
        device=DEVICE,val_batch=VAL_BATCH,throughput_batch=THROUGHPUT_BATCH,
        num_workers=NUM_WORKERS,warmup=WARMUP,iters=ITERS)
else:
    frame = runner.collect_session(SESSION_ID)
display(frame)

## 5. So sánh accuracy, calibration và chi phí
Các metric lấy từ evaluator gốc và toàn bộ 3.501 ảnh val. Δ macro-F1 tính bằng điểm phần trăm.
Chi phí tương đối là p50 phương pháp / p50 I00; số latency dùng đúng pipeline đã cho ra metric.
Chọn ứng viên bằng macro-F1, hòa ưu tiên NLL nhỏ hơn rồi p95 nhỏ hơn.
Nhiệt độ được fit trên chính val: không suy diễn rằng ECE test chắc chắn cải thiện.

In [ ]:
frame = runner.collect_session(SESSION_ID)
columns = ["method","status","macro_f1","top1","ece","nll","delta_macro_f1_pp",
           "latency_p50_ms","latency_p95_ms","latency_p99_ms","relative_p50_cost","throughput_images_per_s"]
display(frame[[c for c in columns if c in frame]])
output = runner.session_paths(SESSION_ID)
if (output / "temperature.json").exists():
    display(json.loads((output / "temperature.json").read_text()))
if (output / "baseline_equivalence.json").exists():
    display(json.loads((output / "baseline_equivalence.json").read_text()))

## 6. Tradeoff và reliability diagrams

In [ ]:
if (output / "I07/summary.json").exists():
    figures = runner.plot_session(SESSION_ID)
    import inference_plots
    inference_plots.plot_tradeoff(SESSION_ID)
    for name in ("f1_vs_latency_p95_detail.png","reliability_before_after.png"):
        path = figures / name
        if path.exists(): display(Image(filename=str(path)))

## 7. F1/recall từng lớp và sample latency

In [ ]:
if (output / "per_class_val.csv").exists() and (output / "per_class_val.csv").stat().st_size > 1:
    per_class = pd.read_csv(output / "per_class_val.csv")
    display(per_class.pivot(index="class_name",columns="method",values=["f1","recall"]))
if (output / "latency_results.csv").exists() and (output / "latency_results.csv").stat().st_size > 1:
    display(pd.read_csv(output / "latency_results.csv"))
print("Mỗi Ixx/latency.json giữ đủ các samples_ms và protocol.")

## 8. Kết luận và bước tiếp theo
Điều kiện: I00 + ít nhất 4 phương pháp khả thi đã đánh giá; calibration trước/sau;
latency đúng protocol, hình tradeoff và quyết định theo val.
Ứng viên realtime cần p95<=100 ms ở batch 1 trong phạm vi đã khai báo.
Phần 10: khóa training/inference/preprocessing/seed/baseline trước test; final và baseline >=3 seed.
Giữ cả calibrated và uncalibrated khi xuất chung kết để so sánh ECE. Chưa đánh giá test trong notebook này.

In [ ]:
if (output / "selection.json").exists():
    selection = json.loads((output / "selection.json").read_text())
    display(selection)
    display(Markdown((output / "inference_report.md").read_text()))
    print("Phần 9:","ĐỦ THÍ NGHIỆM; bổ sung nhận xét" if selection["section9_complete"] else "CHƯA HOÀN TẤT")
else:
    print("Chưa có phiên chạy; bật RUN_QUALITY/RUN_LATENCY trên host GPU.")